# Adding Heiki's location, time and owner annotation
## Extract semantic type annotation from Heiki's csv

In [2]:
import pandas as pd
import sqlite3

### Read csv file in

In [3]:
filename_heiki = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\aeg_koht_elusus\\v11_tabel_m3.csv"

In [4]:
df_raw = pd.read_csv(filename_heiki, 
                       delimiter = ';', 
                       names = ['id','verb','verb_compound','obl_root','obl_case','obl_k','ner_loc','ner_per','ner_org','timex','phrase_type','count'])

In [5]:
df_raw

,id,verb,verb_compound,obl_root,obl_case,obl_k,ner_loc,ner_per,ner_org,timex,phrase_type,count
0,1,olema,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2308619
1,2,toimuma,NaN,lõpp,in,NaN,NaN,NaN,NaN,match,ajam,331
2,3,toimuma,NaN,1.,<käändumatu>,NaN,NaN,NaN,NaN,NaN,NaN,103
3,4,saama,pihta,keel,all,NaN,NaN,NaN,NaN,NaN,NaN,1
4,5,kulmineeruma,NaN,purukspeksmine,kom,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4649808,9844618,minema,maha,jooma,<käändumatu>,NaN,NaN,NaN,NaN,NaN,NaN,1
4649809,9844629,hüppama,NaN,varb,adit,peale,NaN,NaN,NaN,NaN,koham,1
4649810,9844634,jõudma,NaN,milla,gen,NaN,NaN,NaN,NaN,NaN,NaN,1
4649811,9844649,hakkama,NaN,mis,nom,peale,NaN,NaN,NaN,NaN,NaN,1


In [6]:
#what unique semantic tags there are
df_raw['phrase_type'].unique()

array([nan, 'ajam', 'koham', 'tundmatum', 'puuviga', 'omajam'],
      dtype=object)

### Extract info: verb, verb_compound, obl_root, obl_case, phrase_type

In [7]:
#filter for semantic type tags for dependents in spatial cases
df_tags = df_raw.loc[(df_raw['phrase_type'].isin(['ajam', 'koham', 'omajam'])) & (df_raw['obl_case'].isin(['in', 'ill', 'adit', 'el', 'ad', 'all', 'abl']))]

In [8]:
#leave out unnecessary columns
df_needed = df_tags[['verb', 'verb_compound', 'obl_root', 'obl_case', 'phrase_type']]
df_needed

,verb,verb_compound,obl_root,obl_case,phrase_type
1,toimuma,NaN,lõpp,in,ajam
7,viilima,NaN,tund,el,ajam
26,kutsuma,NaN,elu,adit,koham
27,tulema,NaN,toim,adit,koham
38,loobuma,NaN,aeg,ad,ajam
...,...,...,...,...,...
4649780,sattuma,NaN,veiniuim,in,koham
4649789,käima,NaN,Kapa-Kohila,in,koham
4649791,käima,NaN,mio,in,koham
4649795,saama,kokku,kodu,adit,koham


In [9]:
first10 = df_needed[:10]
first10 = first10.fillna('')
first10

,verb,verb_compound,obl_root,obl_case,phrase_type
1,toimuma,,lõpp,in,ajam
7,viilima,,tund,el,ajam
26,kutsuma,,elu,adit,koham
27,tulema,,toim,adit,koham
38,loobuma,,aeg,ad,ajam
48,jooma,,hommik,el,ajam
61,helistama,,hommik,el,ajam
67,täitma,,aasta,ad,ajam
73,ootama,,poi,ill,koham
74,lõpetama,,pedagoogikaülikool,in,koham


### Put results in database table

In [10]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

In [20]:
#connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

#drop table if it exists
cursor.execute("DROP TABLE IF EXISTS heiki_semtypes")

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS heiki_semtypes (
        verb TEXT,
        verb_compound TEXT,
        obl_root TEXT,
        obl_case TEXT,
        phrase_type TEXT
    )
""")

# Step 2: Insert from the dataframe into the database table
first10.to_sql("heiki_semtypes", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()

### Add phrase_type column to spatial_obl table

In [33]:
#add new column to spatial_obl for heiki's annotation
conn = sqlite3.connect(filename)
cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE spatial_obl
    ADD COLUMN heiki_tag TEXT;
""")

In [ ]:
#Connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

cursor.execute("""
    UPDATE spatial_obl
    SET heiki_tag = (
        SELECT phrase_type
        FROM heiki_semtypes
        WHERE 
            heiki_semtypes.verb = spatial_obl.verb
            AND heiki_semtypes.verb_compound = spatial_obl.verb_compound
            AND heiki_semtypes.obl_root = spatial_obl.lemma
            AND heiki_semtypes.obl_case = spatial_obl.morph_case
);
""")

# Commit and close
conn.commit()
conn.close()

In [ ]:
#Connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

cursor.execute("""
    WITH matched AS (
        SELECT s.id AS spatial_id, h.phrase_type
        FROM spatial_obl s
        JOIN heiki_semtypes h
        ON h.verb = s.verb
            AND h.verb_compound = s.verb_compound
            AND h.obl_root = s.lemma
            AND h.obl_case = s.morph_case
    )
    UPDATE spatial_obl
    SET heiki_tag = (
        SELECT phrase_type FROM matched WHERE matched.spatial_id = spatial_obl.rowid
    );
""")

# Commit and close
conn.commit()
conn.close()